# 04 - Invoke Multi-Model Endpoint (Standalone)

This notebook demonstrates how to invoke specific user models on the `hyper-personalization-models` endpoint.

Each model predicts `repeat_purchase` (0/1) based on features:
- `product_encoded` (int): Encoded product ID
- `price` (float): Product price
- `quantity` (int): Quantity purchased
- `rating` (int): User rating 1-5
- `day_of_week` (int): 0=Monday, 6=Sunday
- `month` (int): 1-12

> **Note:** Features are StandardScaler-transformed during training, so raw values are passed and the model handles them.
> In production, you'd include the scaler in a PyFunc wrapper. For this demo, we pass scaled sample values.

In [0]:
from databricks.sdk import WorkspaceClient
import json

w = WorkspaceClient()

# Parameters
dbutils.widgets.text("endpoint_name", "hyper-personalization-models", "Endpoint Name")
dbutils.widgets.text("user_ids", "user_alice,user_bob,user_carol,user_dave,user_eve", "User IDs (comma-separated)")

ENDPOINT_NAME = dbutils.widgets.get("endpoint_name")
USER_IDS = [u.strip() for u in dbutils.widgets.get("user_ids").split(",")]

print(f"Endpoint: {ENDPOINT_NAME}")
print(f"Available models: {[f'{uid.replace(chr(95), chr(45))}-model' for uid in USER_IDS]}")

In [0]:
def invoke_model(endpoint_name, served_model_name, input_data):
    """Invoke a specific model on the multi-model endpoint."""
    from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
    
    response = w.serving_endpoints.query(
        name=endpoint_name,
        dataframe_records=input_data,
        extra_params={"served_model_name": served_model_name},
    )
    return response.predictions

print("Invocation helper function ready")

## Invoke Each User Model
We'll send sample feature vectors to each model. The features represent (after scaling):
`[product_encoded, price, quantity, rating, day_of_week, month]`

In [0]:
# Sample input data - 6 features per record
# [product_encoded, price, quantity, rating, day_of_week, month]

# Alice: Electronics - predicting if she'll repurchase a laptop (high price, high rating)
alice_payload = [
    {"product_encoded": 0, "price": 1099.99, "quantity": 1, "rating": 5, "day_of_week": 2, "month": 3},
    {"product_encoded": 5, "price": 29.99, "quantity": 3, "rating": 4, "day_of_week": 5, "month": 7},
]

print("=" * 60)
print("ALICE (Electronics) - user-alice-model")
print("=" * 60)
try:
    preds = invoke_model(ENDPOINT_NAME, "user-alice-model", alice_payload)
    for i, (inp, pred) in enumerate(zip(alice_payload, preds)):
        label = "WILL REPURCHASE" if pred == 1 else "WON'T REPURCHASE"
        print(f"  Input {i+1}: price=${inp['price']}, rating={inp['rating']}, qty={inp['quantity']} -> {label}")
except Exception as e:
    print(f"  Error: {e}")

In [0]:
# Bob: Fitness - predicting repurchase of protein powder (consumable) vs expensive equipment
bob_payload = [
    {"product_encoded": 0, "price": 39.99, "quantity": 2, "rating": 5, "day_of_week": 1, "month": 1},
    {"product_encoded": 3, "price": 179.99, "quantity": 1, "rating": 3, "day_of_week": 6, "month": 11},
]

print("\n" + "=" * 60)
print("BOB (Fitness) - user-bob-model")
print("=" * 60)
try:
    preds = invoke_model(ENDPOINT_NAME, "user-bob-model", bob_payload)
    for i, (inp, pred) in enumerate(zip(bob_payload, preds)):
        label = "WILL REPURCHASE" if pred == 1 else "WON'T REPURCHASE"
        print(f"  Input {i+1}: price=${inp['price']}, rating={inp['rating']}, qty={inp['quantity']} -> {label}")
except Exception as e:
    print(f"  Error: {e}")

In [0]:
# Carol: Kitchen - predicting repurchase of spices (cheap, consumable) vs expensive knife
carol_payload = [
    {"product_encoded": 1, "price": 24.99, "quantity": 3, "rating": 5, "day_of_week": 3, "month": 5},
    {"product_encoded": 6, "price": 189.99, "quantity": 1, "rating": 2, "day_of_week": 0, "month": 12},
]

print("\n" + "=" * 60)
print("CAROL (Kitchen) - user-carol-model")
print("=" * 60)
try:
    preds = invoke_model(ENDPOINT_NAME, "user-carol-model", carol_payload)
    for i, (inp, pred) in enumerate(zip(carol_payload, preds)):
        label = "WILL REPURCHASE" if pred == 1 else "WON'T REPURCHASE"
        print(f"  Input {i+1}: price=${inp['price']}, rating={inp['rating']}, qty={inp['quantity']} -> {label}")
except Exception as e:
    print(f"  Error: {e}")

In [0]:
# Dave: Books - predicting repurchase of cheap novels vs expensive e-reader
dave_payload = [
    {"product_encoded": 0, "price": 14.99, "quantity": 4, "rating": 5, "day_of_week": 4, "month": 8},
    {"product_encoded": 1, "price": 229.99, "quantity": 1, "rating": 3, "day_of_week": 2, "month": 2},
]

print("\n" + "=" * 60)
print("DAVE (Books) - user-dave-model")
print("=" * 60)
try:
    preds = invoke_model(ENDPOINT_NAME, "user-dave-model", dave_payload)
    for i, (inp, pred) in enumerate(zip(dave_payload, preds)):
        label = "WILL REPURCHASE" if pred == 1 else "WON'T REPURCHASE"
        print(f"  Input {i+1}: price=${inp['price']}, rating={inp['rating']}, qty={inp['quantity']} -> {label}")
except Exception as e:
    print(f"  Error: {e}")

In [0]:
# Eve: Fashion - predicting repurchase of luxury items
eve_payload = [
    {"product_encoded": 6, "price": 129.99, "quantity": 2, "rating": 5, "day_of_week": 6, "month": 6},
    {"product_encoded": 0, "price": 799.99, "quantity": 1, "rating": 2, "day_of_week": 1, "month": 9},
]

print("\n" + "=" * 60)
print("EVE (Fashion) - user-eve-model")
print("=" * 60)
try:
    preds = invoke_model(ENDPOINT_NAME, "user-eve-model", eve_payload)
    for i, (inp, pred) in enumerate(zip(eve_payload, preds)):
        label = "WILL REPURCHASE" if pred == 1 else "WON'T REPURCHASE"
        print(f"  Input {i+1}: price=${inp['price']}, rating={inp['rating']}, qty={inp['quantity']} -> {label}")
except Exception as e:
    print(f"  Error: {e}")

## Alternative: REST API Invocation
You can also invoke the endpoint via REST API (e.g., from external applications):

In [0]:
# REST API example (for reference - can be used from any HTTP client)
import os

host = w.config.host
token = w.config.token

curl_example = f"""
curl -X POST {host}/serving-endpoints/{ENDPOINT_NAME}/invocations \\
  -H "Authorization: Bearer $DATABRICKS_TOKEN" \\
  -H "Content-Type: application/json" \\
  -d '{{
    "dataframe_records": [
      {{"product_encoded": 0, "price": 899.99, "quantity": 1, "rating": 5, "day_of_week": 2, "month": 3}}
    ],
    "params": {{
      "served_model_name": "user-alice-model"
    }}
  }}'
"""

print("REST API curl example:")
print(curl_example)

In [0]:
import random
import time

# Run 200 invocations hitting a random model each time
# This generates enough traffic so the metrics API reports non-zero request counts
NUM_INVOCATIONS = 200
model_names = [f"{uid.replace('_', '-')}-model" for uid in USER_IDS]

# Sample payload (features don't matter here - just generating traffic)
sample_payload = [{"product_encoded": 0, "price": 50.0, "quantity": 2, "rating": 4, "day_of_week": 3, "month": 6}]

print(f"Running {NUM_INVOCATIONS} invocations across {len(model_names)} models...\n")

invocation_counts = {m: 0 for m in model_names}
errors = 0

for i in range(NUM_INVOCATIONS):
    target_model = random.choice(model_names)
    try:
        invoke_model(ENDPOINT_NAME, target_model, sample_payload)
        invocation_counts[target_model] += 1
    except Exception as e:
        errors += 1
    
    if (i + 1) % 50 == 0:
        print(f"  Completed {i + 1}/{NUM_INVOCATIONS} invocations...")

print(f"\n\u2705 Done! {NUM_INVOCATIONS - errors} successful, {errors} errors")
print(f"\nInvocations per model:")
for model, count in sorted(invocation_counts.items()):
    print(f"  {model}: {count}")

# Wait 3 minutes for metrics to propagate
# The Databricks metrics API aggregates on ~1-5 min intervals,
# so we need to wait before the request_count_total reflects our traffic
print(f"\n\u23f3 Waiting 3 minutes for metrics to propagate...")
time.sleep(180)
print("\u2705 Ready to fetch metrics!")

In [0]:
import requests
import re
import pandas as pd
import matplotlib.pyplot as plt

# Fetch endpoint metrics (Prometheus format)
host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

response = requests.get(
    f"https://{host}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/metrics",
    headers={"Authorization": f"Bearer {token}"},
)

if response.status_code != 200:
    print(f"Error fetching metrics: {response.status_code} - {response.text[:500]}")
else:
    raw_metrics = response.text
    lines = [l for l in raw_metrics.strip().split("\n") if not l.startswith("#") and l.strip()]
    print(f"\U0001f4e1 Metrics fetched: {len(lines)} data points\n")
    
    # Parse Prometheus format:
    # metric_name{label1="val1",label2="val2",} value timestamp
    metrics_data = []
    for line in lines:
        # Match: metric{labels} value [timestamp]
        match = re.match(r'(\w+)\{(.*)\}\s+([\d.eE+-]+)', line)
        if match:
            metric_name = match.group(1)
            labels_str = match.group(2)
            value = float(match.group(3))
            
            # Extract servedModelName (the actual label used by Databricks)
            model_match = re.search(r'servedModelName="([^"]+)"', labels_str)
            model_name = model_match.group(1) if model_match else "unknown"
            
            metrics_data.append({
                "metric": metric_name,
                "model": model_name,
                "value": value,
            })
    
    metrics_df = pd.DataFrame(metrics_data)
    
    if not metrics_df.empty:
        print("=" * 70)
        print("ENDPOINT METRICS PER SERVED MODEL")
        print("=" * 70)
        
        for metric in sorted(metrics_df["metric"].unique()):
            subset = metrics_df[metrics_df["metric"] == metric]
            print(f"\n\U0001f4ca {metric}:")
            for _, row in subset.iterrows():
                print(f"   {row['model']:25s} = {row['value']:.4f}")
        
        # Visualize per-model metrics
        fig, axes = plt.subplots(2, 2, figsize=(14, 9))
        
        # CPU usage per model
        cpu = metrics_df[metrics_df["metric"] == "cpu_usage_percentage"].sort_values("model")
        if not cpu.empty:
            axes[0, 0].barh(cpu["model"], cpu["value"], color="steelblue")
            axes[0, 0].set_title("CPU Usage (%)")
            axes[0, 0].set_xlabel("%")
        
        # Memory usage per model
        mem = metrics_df[metrics_df["metric"] == "mem_usage_percentage"].sort_values("model")
        if not mem.empty:
            axes[0, 1].barh(mem["model"], mem["value"], color="coral")
            axes[0, 1].set_title("Memory Usage (%)")
            axes[0, 1].set_xlabel("%")
        
        # Provisioned concurrency per model
        prov = metrics_df[metrics_df["metric"] == "provisioned_concurrent_requests_total"].sort_values("model")
        if not prov.empty:
            axes[1, 0].barh(prov["model"], prov["value"], color="seagreen")
            axes[1, 0].set_title("Provisioned Concurrency")
            axes[1, 0].set_xlabel("Concurrent Requests")
        
        # Request count per model
        req = metrics_df[metrics_df["metric"] == "request_count_total"].sort_values("model")
        if not req.empty:
            axes[1, 1].barh(req["model"], req["value"], color="mediumpurple")
            axes[1, 1].set_title("Total Request Count")
            axes[1, 1].set_xlabel("Requests")
        
        for ax in axes.flat:
            ax.grid(axis="x", alpha=0.3)
        
        plt.suptitle(f"Endpoint: {ENDPOINT_NAME}", fontsize=13, fontweight="bold")
        plt.tight_layout()
        plt.show()
        
        # Full pivot table
        print("\n")
        pivot = metrics_df.pivot_table(index="model", columns="metric", values="value", aggfunc="sum").round(4)
        display(pivot.reset_index())